# 08 - Summarizing and combining

Last time we got a file off the disk and into a table we could trust. This time we do the two things
that turn a table into an answer:

- **Summarizing** - questions about *groups* of rows rather than single ones
- **Combining** - putting two tables together, and checking that it worked

The notebook is organized by topic, and the sections are independent enough to be read in any order
later:

| Section | |
|---|---|
| 1. Grouping and aggregating | `groupby`, `.agg()`, and one row per group |
| 2. The index of a grouped result | `set_index` and `reset_index` |
| 3. Transforming within groups | `.transform()`, `ffill`, `diff`, `pct_change` - one row per *original* row |
| 4. Working with dates | `to_datetime`, date formats, the `.dt` accessor |
| 5. Resampling a time series | `resample`, up and down |
| 6. Combining tables | `concat`, duplicates, `merge`, and validating a merge |
| 7. Applying your own functions | `.apply()` on a groupby |

> 📝 **Note:** Two cells in this notebook are *meant* to fail, and are marked with a comment
naming the error - `# KeyError` and `# MergeError`. Several other cells produce answers that are
**wrong without failing**; the text says so where that happens. Run cells one at a time rather than
using Run All.

As always, we start by importing what we need and loading our own data.

In [ ]:
import numpy as np
import pandas as pd

Here is the function we finished with last time. Nothing in it is new - it reads the file, drops the
rows with no emissions figure, and adds emissions per person. Last week that was a notebook's worth
of work; this week it is one line, which is the whole reason for having written it.

In [ ]:
def load_emissions(path):
    """
    Read the emissions file and return it ready to use.

    Parameters
    ----------
    path : str
        Path to the emissions CSV file.

    Returns
    -------
    DataFrame
        One row per country per year, with rows missing total emissions
        dropped and emissions per person added as a column.
    """
    emissions = pd.read_csv(path)

    emissions = emissions.dropna(subset=["co2_total"])
    emissions["co2_pc"] = emissions["co2_total"] * 1_000_000 / emissions["population"]

    return emissions


co2 = load_emissions("../data/co2_emissions.csv")

print(co2.shape)
co2.head(3)

## 1. Grouping and aggregating

Here is a question the table looks like it can answer. **How much CO₂ did the world emit each
year?**

Every row is one entity in one year. So: take all the rows for a year, add up their emissions, and
do that for each year. That is a loop over years, and you could write one - you have written harder
loops than that. pandas has a shorter way, called `groupby`.

(`.tail(3)` below shows the **last** three rows, the way `.head(3)` shows the first three.)

In [ ]:
co2.groupby("year")["co2_total"].sum().tail(3)

Three moves in one line: **split** the rows into groups by their `year`, **apply** `sum` to the
`co2_total` column of each group, and **combine** the answers into one result. That is the whole
idea, and it has a name - **split-apply-combine**. Every `groupby` you will ever write is those three
steps; the only things that change are what you split on and what you apply.

> ⚠️ **Warning:** That answer is **wrong**, by a factor of more than eight. The table contains rows
like `World` and `East Asia & Pacific`, which are groupings of countries, so summing every row counts
the same emissions several times over. We come back to it in section 6, once we can bring in a second
table that says which entities are countries. Until then, treat every total in this notebook as a
demonstration of the method rather than a fact about the world.

### The grouped object computes nothing

It is worth seeing what `groupby` actually returns, because it is not a table.

In [ ]:
by_year = co2.groupby("year")

by_year

A `DataFrameGroupBy`. It knows which rows belong to which group and has computed nothing at all - it
is waiting to be told what to apply. That is why the two halves are always written together:
`.groupby("year")` on its own is a question with no verb.

Pick a column first and you get one answer per group.

In [ ]:
co2.groupby("year")["co2_pc"].mean().tail(3)

Leave the column out and pandas applies the function to every numeric column it can, which is
occasionally what you want and usually noise.

You can group on more than one thing by passing a **list** of column names. We have nothing sensible
to use as a second key yet - the column we want says which region each country is in, and it lives in
a file we have not opened - so that waits for section 6.

### Counting: `.size()` and `.count()`

Two ways to count, and the difference matters.

- `.size()` counts **rows** in each group.
- `.count()` counts **non-missing values** in a column of each group.

In [ ]:
counts = pd.DataFrame({
    "rows": co2.groupby("year").size(),
    "renew_energy": co2.groupby("year")["renew_energy"].count(),
})

counts.tail(5)

Every year has the same 246 rows, and until 2020 every row has a renewable energy figure. Then 203,
then 66, then **none at all**. That is the publication lag we found last time, seen from a different
direction: the rows are all there, the values are not.

Reach for `.size()` when you want to know how big a group is, and `.count()` when you want to know how
much of it you can actually use.

### Several statistics at once: `.agg()`

`.mean()` gives you one statistic. Real summary tables want several. `.agg()` takes a **list of
function names, written as strings**, and gives you a column for each.

In [ ]:
co2.groupby("year")["co2_pc"].agg(["mean", "median", "max"]).tail(3)

These are the aggregation functions worth knowing. All of them collapse a group to a single value,
and all of them can be written either as a method or as a string inside `.agg()`.

| | |
|---|---|
| `"sum"` | total |
| `"mean"`, `"median"` | average, middle value |
| `"min"`, `"max"` | smallest, largest |
| `"std"`, `"var"` | standard deviation, variance |
| `"count"` | non-missing values |
| `"size"` | rows, missing or not |
| `"first"`, `"last"` | first and last value in the group |
| `"nunique"` | how many distinct values |

The list form works when every statistic is of the **same column**. More often you want different
statistics of different columns, and then there is a second form: name each output column, and say
what it is made of.

```python
grouped.agg(
    output_name = ("input_column", "function"),
    ...
)
```

In [ ]:
summary = co2.groupby("year").agg(
    entities=("co2_total", "count"),
    total_co2=("co2_total", "sum"),
    mean_pc=("co2_pc", "mean"),
)

summary.tail(3)

One line per output column, each saying exactly where it came from. This is the form to reach for
when you are building a table somebody else will read, because the column names are yours rather than
pandas' guesses.

> 💡 **Tip:** `.round(2)` on the result rounds every number in it, which usually makes a
summary table far easier to read. Round for **display**, at the end - not in the middle of a
calculation, where you would be throwing away precision you still need.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")</code></pre>

<p>Build one summary table with a row per year and three named columns: the number of entities that
have a <code>gdp_pc</code> figure, the average <code>gdp_pc</code>, and the highest
<code>urban</code> share. Display the last five years, rounded to one decimal.</p>
</div>

## 2. The index of a grouped result

Look closely at what came back from those last few cells. The years are not in a column. They are
down the left-hand side, where the row numbers usually go.

That is because they **are** the index now. `groupby` puts whatever you grouped on into the index of
the result, which makes sense: the group label is what identifies the row.

In [ ]:
totals = co2.groupby("year")["co2_total"].sum()

print(type(totals))
print(totals.index)

A `Series`, indexed by year. That is often exactly what you want. Index alignment means you can divide
one grouped result by another and the years line themselves up, and plotting a Series puts the index
along the horizontal axis without being asked.

But it is a nuisance the moment you want to treat the result as an ordinary table - filter it, join
it, or write it to a file - because **`year` is not a column**, and asking for it as one fails.

In [ ]:
# KeyError
totals["year"]

### `reset_index` moves the index back into a column

In [ ]:
totals_df = totals.reset_index()

print(type(totals_df))
totals_df.tail(3)

Now it is a DataFrame with two ordinary columns, and everything from last time works on it again.
`reset_index()` is the most common thing to write after a `groupby`, and when a grouped result is
fighting you, it is usually the answer.

### `set_index` goes the other way

`set_index` takes a column and makes it the index.

In [ ]:
totals_by_year = totals_df.set_index("year")

totals_by_year.loc[2023]

Which raises the obvious question: why would you ever do that on purpose?

Because an index is not decoration - it is what pandas **aligns on**, as we saw when two Series with
different labels were added together. A table indexed by something meaningful can be looked up by
label with `.loc`, as above; it lines up automatically with any other table indexed the same way; and
if the index holds dates, it can be grouped by time, which is what section 5 is about.

> 📝 **Note:** `reset_index()` takes a `drop` parameter. `reset_index(drop=True)` throws the
old index away instead of turning it into a column, which is what you want when the index is
meaningless row numbers rather than something you care about. It turns up in section 6.

## 3. Transforming within groups

Everything in section 1 **collapsed** the table: 5 904 rows in, 24 rows out, one per group.

Often that is not what you want. You want the group's answer put back **beside every row it came
from**, so that a row can be compared against its own group. Norway's emissions per person in 2020 are
only interesting next to Norway's usual level.

The methods in this section all do that. They take a group and give back **one value per original
row**, so their results can be assigned straight into a column.

In [ ]:
print("mean       :", co2.groupby("country")["co2_pc"].mean().shape)
print("transform  :", co2.groupby("country")["co2_pc"].transform("mean").shape)

246 against 5 904. `.mean()` gave one number per country; `.transform("mean")` gave every row its own
country's number, repeated as many times as that country has rows. **Same answer, different shape.**

> ⚠️ **Warning:** If you assign a grouped result to a column and get a wall of `NaN`, you almost
certainly used an aggregation where you needed a transformation: pandas aligned 246 country names
against 5 904 row numbers, found no matches, and filled the lot with missing values. That is index
alignment doing exactly what it promised to do.

### The methods

`.transform()` takes any of the aggregation names from section 1 and broadcasts the result back. The
rest are methods in their own right, and they are the reason this section exists: each one compares a
row against its **neighbors** rather than against a summary.

| | |
|---|---|
| `.transform("mean")` | the group's statistic, repeated on every row of the group |
| `.diff()` | this row minus the previous row |
| `.pct_change()` | the same, as a fraction of the previous row |
| `.shift(1)` | the previous row's value, moved down one |
| `.cumsum()` | running total down the group |
| `.ffill()` | fill a hole with the last known value **before** it |
| `.bfill()` | fill a hole with the next known value **after** it |

Because several of these look at the row before or after, the rows have to be **in order**. Our panel
is one series per country stacked on top of the next, so we sort by country and year first.

In [ ]:
panel = co2.sort_values(["country", "year"]).copy()

panel["country_mean_pc"] = panel.groupby("country")["co2_pc"].transform("mean")
panel["previous"] = panel.groupby("country")["co2_total"].shift(1)
panel["change"] = panel.groupby("country")["co2_total"].diff()
panel["pct_change"] = panel.groupby("country")["co2_total"].pct_change() * 100

panel[panel["country"] == "Norway"][
    ["year", "co2_total", "previous", "change", "pct_change", "country_mean_pc"]
].tail(4).round(2)

Four different questions about Norway, in one table. `previous` is last year's figure moved down a
row; `change` is the difference; `pct_change` is that difference as a percentage; `country_mean_pc` is
Norway's own 24-year average per person, repeated on every Norwegian row.

That last one is the pattern worth remembering: **compute a baseline per group, then measure every row
against it.** Temperature anomalies, deviations from trend, shares of a total and index numbers are
all these same two lines.

In [ ]:
year_2020 = panel[panel["year"] == 2020]

print("entities with a 2020 figure:", len(year_2020))
print("emissions fell in 2020:     ", (year_2020["change"] < 0).sum())

187 of 246 emitted less in 2020 than in 2019 - the pandemic, visible in one grouped `.diff()`.

### Filling gaps

`ffill` and `bfill` belong to the same family and are worth their own demonstration, because they are
a **decision** and not a repair. We know `renew_energy` stops being published in the most recent
years. A forward fill carries each country's last observed share forward into them.

In [ ]:
panel["renew_filled"] = panel.groupby("country")["renew_energy"].ffill()

print("missing before:", panel["renew_energy"].isna().sum())
print("missing after: ", panel["renew_filled"].isna().sum())

panel[panel["country"] == "Norway"][["year", "renew_energy", "renew_filled"]].tail(4)

471 holes down to 2, and Norway's 2021 figure of 61.4 copied into 2022 and 2023.

**And now the part you have to decide rather than run.** A forward fill asserts that the value did not
change. Over one year, for a share that moves slowly, that is defensible. Over three years, for exactly
the years everyone wants to look at, it invents a flat line that never happened - and nothing
downstream will ever tell you the number was made up. Filling is a claim about the world, not a
tidying step.

`bfill` is the same method pointing the other way, and it earns its keep when a label is recorded only
once. If a file gives each country's region in its final year only, one `bfill` within country carries
it back over every earlier year.

### The group is not optional

Every method in this section was written with a `groupby` in front of it. Here is what happens
without one.

In [ ]:
print("ffill, grouped:  ", panel.groupby("country")["renew_energy"].ffill().isna().sum(), "missing")
print("ffill, ungrouped:", panel["renew_energy"].ffill().isna().sum(), "missing")
print()

ungrouped_change = panel["co2_total"].diff()
wrong = panel["change"].isna() & ungrouped_change.notna()

print("diff, rows where ungrouped disagrees with grouped:", wrong.sum())
panel[wrong][["country", "year", "co2_total"]].assign(ungrouped=ungrouped_change[wrong]).head(3).round(2)

Two failures of the same kind, and neither raises anything.

The ungrouped `ffill` leaves **zero** missing values, which looks better and is much worse: having run
out of Norwegian values it carried straight on into the next country's rows and filled those with
Norway's number.

The ungrouped `diff` disagrees on **245** rows - one for each country after the first. Every one of
them is a country's first year, where the grouped version correctly says `NaN` and the ungrouped
version reports this country's first year minus the *previous country's* last year. Albania emitted
3.23 million tonnes in 2000, and the ungrouped column records that as a fall of 248.41, which is not a
fact about Albania.

> ⚠️ **Warning:** Every method in this section looks at the row before or the row after, and none of
them knows where one country ends and the next begins. **Group first** - and the group is whatever the
rows are a series *within*.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")
co2 = co2.sort_values(["country", "year"])</code></pre>

<p>Add a column giving each country's year-on-year <b>percentage</b> change in
<code>co2_total</code>, and display the five largest single-year rises in the whole table.</p>
</div>

## 4. Working with dates

A great deal of real data is a **time series**: the same thing measured over and over, with a date
attached. Before you can group, filter or plot by time, pandas has to know that the date column is a
date and not a piece of text.

New dataset for this section and the next: the daily closing level of the **NASDAQ Composite** stock
index, from 1971 to 2025.

In [ ]:
nasdaq = pd.read_csv("../data/NASDAQ.csv")

print(nasdaq.shape)
print(nasdaq.dtypes)
nasdaq.head(3)

13 842 trading days and two columns. And `Date` is **`str`** - as far as pandas is concerned,
`"1971-02-05"` is a piece of text like `"Bergen"`.

### `pd.to_datetime`, and always say the format

`pd.to_datetime` converts text into real dates. It can guess the format, and you should not let it.

In [ ]:
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"], format="%Y-%m-%d")

print(nasdaq["Date"].dtype)
print(nasdaq["Date"].min(), "to", nasdaq["Date"].max())

`datetime64` - a point in time, which pandas can sort, compare and do arithmetic on.

The `format` string describes how the date is *written*, using a code for each piece:

| Code | Means | Example |
|---|---|---|
| `%Y` | four-digit year | `2024` |
| `%y` | two-digit year | `24` |
| `%m` | month as a number | `03` |
| `%b` | short month name | `Mar` |
| `%B` | full month name | `March` |
| `%d` | day of the month | `09` |
| `%H` | hour, 24-hour clock | `14` |
| `%M` | minute | `30` |
| `%S` | second | `05` |

Everything else in the string is taken literally, so `"%d/%m/%Y"` matches `09/03/2024`,
`"%d.%m.%Y"` matches `09.03.2024`, and `"%B %d, %Y"` matches `March 09, 2024`.

### Why the format is not optional

In [ ]:
visits = pd.DataFrame({
    "date": ["02/01/2021", "06/05/2021", "11/03/2021"],
    "patients": [14, 9, 22],
})

print("guessed:  ", list(pd.to_datetime(visits["date"]).dt.date))
print("told:     ", list(pd.to_datetime(visits["date"], format="%d/%m/%Y").dt.date))

Those are European dates - 2 January, 6 May, 11 March. Left to guess, pandas reads them the American
way round and returns **1 February, 5 June and 3 November** instead.

No error, and no warning. Every day number here is 12 or below, so both readings are valid calendar
dates and there is nothing for pandas to complain about. The dates are simply wrong, and every
grouping, filter and plot built on them afterwards is wrong too.

> ⚠️ **Warning:** Always pass `format=`, and always check a few converted values against the
originals. This is one of the few mistakes in the whole course that is completely invisible.

### `.dt` gets the pieces out

A real date knows what year, month and day it is made of. The **`.dt` accessor** is how you ask for
one piece at a time: `.dt.year`, `.dt.month`, `.dt.day`, `.dt.quarter`, `.dt.date`, and
`.dt.day_name()` for the weekday as text.

In [ ]:
nasdaq["year"] = nasdaq["Date"].dt.year
nasdaq["month"] = nasdaq["Date"].dt.month
nasdaq["weekday"] = nasdaq["Date"].dt.day_name()

nasdaq.head(3)

Each of those returns an ordinary column, so everything from section 1 applies to them - `groupby`
included.

### Filtering by date

Real dates compare properly, so filtering to a period is a boolean mask like any other. The
boundaries can be written as strings and pandas will read them as dates.

In [ ]:
crash = nasdaq[(nasdaq["Date"] >= "2020-02-01") & (nasdaq["Date"] <= "2020-04-30")]

print(len(crash), "trading days")
print("high:", crash["NASDAQ"].max().round(1), " low:", crash["NASDAQ"].min().round(1))

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>nasdaq = pd.read_csv("../data/NASDAQ.csv")
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"], format="%Y-%m-%d")</code></pre>

<p>Using the <code>.dt</code> accessor and a <code>groupby</code>, report the average closing level in
each <b>decade</b>. Display the result.</p>

<p><i>Hint: integer division by 10 and back, <code>year // 10 * 10</code>, turns 1987 into
1980.</i></p>
</div>

## 5. Resampling a time series

Daily data is rarely the resolution you want to report at. `resample` changes the frequency of a time
series: **downsampling** to a coarser one (daily to monthly), or **upsampling** to a finer one
(monthly to daily).

It needs the dates to be the **index**, which is what `set_index` was for, and it wants them in order,
which is what `sort_index` is for.

In [ ]:
prices = nasdaq.set_index("Date").sort_index()["NASDAQ"]

prices.head(3)

### Downsampling

`resample` takes a string saying which period to group into, and then a verb saying what to do with
each period - exactly like `groupby`.

| Code | Period |
|---|---|
| `"D"` | day |
| `"W"` | week |
| `"ME"` | month end |
| `"QE"` | quarter end |
| `"YE"` | year end |

For a price series, the sensible verb is `.last()` - the closing level of the period.

In [ ]:
print("daily rows:  ", len(prices))
print("weekly rows: ", len(prices.resample("W").last()))
print("monthly rows:", len(prices.resample("ME").last()))
print("yearly rows: ", len(prices.resample("YE").last()))

In [ ]:
annual = prices.resample("YE").last()

annual.head(3).round(1)

13 842 daily observations become 55 annual ones. The verb matters and is yours to choose: `.last()`
for a closing level, `.mean()` for an average level, `.max()` for the peak, `.sum()` for a quantity
like traded volume or rainfall. Summing a *price* would be meaningless, and pandas will do it happily
if you ask.

Because the result is an ordinary Series indexed by date, the transformation methods from section 3
work on it directly.

In [ ]:
annual_return = annual.pct_change() * 100

print("worst years:")
print(annual_return.sort_values().head(3).round(1))
print()
print("best years:")
print(annual_return.sort_values(ascending=False).head(3).round(1))

2008 and 2000, the financial crisis and the dot-com crash, then 1974. And 1999, the year before the
dot-com crash, up 86%.

### Upsampling

Going the other way asks for rows that do not exist in the data, and this is where `resample` shows
what it really is. Ask for **every calendar day**.

In [ ]:
daily = prices.resample("D").last()

print("trading days in the file:", len(prices))
print("calendar days produced: ", len(daily))
print("of which empty:         ", daily.isna().sum())
print()
print(daily.loc["2020-03-05":"2020-03-10"].round(1))

20 053 calendar days from 13 842 trading days, and **6 211 of them empty**. Saturday 7 March and
Sunday 8 March 2020 are `NaN` because no trading happened - and those rows now exist, which they did
not before.

That is the point of `resample`, and it is worth stating plainly:

> **`resample` builds its periods out of the calendar, not out of the rows you happen to have.**

A gap in the data becomes a visible `NaN` rather than silently disappearing. If you want the gaps
filled, that is a separate decision and you make it yourself - with the method from section 3.

In [ ]:
print("empty after ffill:", prices.resample("D").last().ffill().isna().sum())

### `resample` and `groupby` are not the same thing

You could imagine doing the monthly figures with `groupby` on `.dt.month` instead. Try it.

In [ ]:
print("groupby month :", len(prices.groupby(prices.index.month).mean()), "rows")
print("resample month:", len(prices.resample("ME").mean()), "rows")
print()
print(prices.groupby(prices.index.month).mean().head(3).round(1))

Twelve rows against 659, and the twelve are nonsense. `.dt.month` returns the number 1 for **every
January in the dataset**, so those groups mix January 1971 with January 2025 - a 55-year span over
which the index went from 100 to 23 419. The average of that is not a fact about anything.

Both behaviors are useful, for different questions:

- **`groupby` on a date part** answers *seasonal* questions: are Januaries usually weak? It is right
  to merge fifty-five Januaries when that is what you are asking about.
- **`resample`** answers questions about a *series through time*: what happened, in order. It keeps
  the periods separate and dated.

If the answer belongs on a chart with time along the bottom, you want `resample`.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>nasdaq = pd.read_csv("../data/NASDAQ.csv")
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"], format="%Y-%m-%d")
prices = nasdaq.set_index("Date").sort_index()["NASDAQ"]</code></pre>

<p>Resample the series to <b>quarter</b> ends, taking the closing level of each quarter, and report
how many quarters the index ended lower than the quarter before.</p>
</div>

## 6. Combining tables

Everything so far has been one file at a time. Real work is rarely one file: a year per file, a
country per file, one file from each of ten colleagues - or, as here, the measurements in one file and
the labels that explain them in another.

There are two ways to put tables together, and which you want depends on what the second table adds:

- **`concat`** adds **rows**: same columns, more observations.
- **`merge`** adds **columns**: same observations, more variables.

### `concat`: stacking

`pd.concat` takes a **list** of DataFrames and returns one.

In [ ]:
north = pd.DataFrame({
    "country": ["Norway", "Sweden", "Denmark"],
    "co2_pc": [7.5, 3.4, 4.6],
})

south = pd.DataFrame({
    "country": ["Spain", "Italy"],
    "co2_pc": [5.0, 5.4],
})

stacked = pd.concat([north, south])

stacked

Five rows, as expected. But look down the left-hand side: **0, 1, 2, 0, 1**. `concat` stacked the
indexes too, so two different rows are both labeled 0 - and the index is what pandas aligns on, so a
duplicated index quietly breaks `.loc` and everything built on it.

In [ ]:
stacked.loc[0]

Two rows returned for one label. The fix is `reset_index` with the `drop` parameter from section 2 -
these row numbers mean nothing, so throw them away rather than keeping them as a column.

In [ ]:
stacked = pd.concat([north, south]).reset_index(drop=True)

stacked

**Write it as one expression, every time.** `pd.concat([...]).reset_index(drop=True)` is the idiom;
the version without it is a bug waiting for somebody to use `.loc`.

If the tables do not have quite the same columns, `concat` keeps all of them and fills the gaps with
`NaN` rather than refusing. That is convenient, and worth checking rather than trusting: a column of
`NaN` you did not expect usually means two files disagree about a column name - `co2_pc` in one and
`CO2_pc` in the other - and `concat` will not tell you.

> 📝 **Note:** `pd.concat([a, b], axis=1)` glues tables side by side instead, matching on the
index. It is occasionally right and usually not: when you want columns from another table, what you
almost always want is a merge.

### Duplicates

Stacking files together is the commonest way to end up with the same row twice - a file listed twice
in a folder, a cell run twice, two colleagues who both sent you January.

`.duplicated()` gives one `True` or `False` per row, marking rows identical to one seen earlier. It is
a boolean mask, so summing it counts them, and `.drop_duplicates()` removes them.

In [ ]:
readings = pd.DataFrame({
    "station": ["Oslo", "Bergen", "Oslo", "Tromsø", "Bergen"],
    "year": [2023, 2023, 2023, 2023, 2023],
    "temp": [6.9, 8.4, 6.9, 3.1, 8.4],
})

print("duplicate rows:", readings.duplicated().sum())
readings.drop_duplicates()

Rows 2 and 4 repeated rows 0 and 1 exactly, and dropping them lost nothing, because the duplicates
**agreed**.

Here is the case that matters.

In [ ]:
readings_2 = pd.DataFrame({
    "station": ["Oslo", "Bergen", "Oslo", "Tromsø"],
    "year": [2023, 2023, 2023, 2023],
    "temp": [6.9, 8.4, 7.4, 3.1],
})

print("duplicate rows:", readings_2.duplicated().sum())
readings_2

**Zero duplicates**, and Oslo is still in there twice - with two different temperatures. No row is a
copy of another, so `.duplicated()` is right to say nothing.

To ask "is any station listed twice?", say which columns identify a row, using `subset`. `keep=False`
marks **every** row involved rather than only the later one, which is what you want when you are
looking at the problem instead of deleting it.

In [ ]:
print("repeated station-year keys:", readings_2.duplicated(subset=["station", "year"]).sum())

readings_2[readings_2.duplicated(subset=["station", "year"], keep=False)]

> ⚠️ **Warning:** An exactly duplicated row is a mistake, and dropping it loses nothing. A repeated
**key** with conflicting values is a question about your data - which figure is right, and why are
there two? `drop_duplicates(subset=["station", "year"])` would silently keep 6.9 and discard 7.4 for
no better reason than that it came first. Find out before you drop.

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>info = pd.read_csv("../data/country_info.csv")</code></pre>

<p>This lookup table should have exactly one row per country. Check both things that could be wrong
with it: how many rows are exact duplicates, and how many <code>code</code> values appear more than
once. Print both counts.</p>
</div>

### `merge`: columns from another table

`merge` matches rows in one table against rows in another on a shared value called a **key**.

Here is the second file - the one that will finally let us fix the total from section 1.

In [ ]:
info = pd.read_csv("../data/country_info.csv")

print(info.shape)
info.head(3)

In [ ]:
info["region"].value_counts()

295 rows, one per entity: a `name`, a three-letter `code`, a `region` and an `incomeLevel`. And there
is the column we need - the World Bank marks its own groupings with the region `"Aggregates"`.

### First, tidy the column names with `rename`

`incomeLevel` is camelCase, and everything in this course is snake_case. `rename` takes a dictionary
of old name to new name and returns a new table.

In [ ]:
info = info.rename(columns={"incomeLevel": "income_level"})

info.columns

`rename` is worth knowing well beyond tidiness. It is how you make two tables agree about what a
column is called, which matters here because our two tables name the same thing differently: the
emissions file calls it `country`, the lookup file calls it `name`.

That gives two ways to merge on it - rename one side so the names match, or tell `merge` about both
names with `left_on` and `right_on`:

```python
info.rename(columns={"name": "country"})     # then merge on="country"
co2.merge(info, left_on="country", right_on="name")   # or say both
```

Use `rename` when the mismatch is a nuisance you want gone for good, and `left_on`/`right_on` when it
is a one-off. We will use both below.

### Look at the keys before you use them

A merge matches keys **exactly**. Not roughly, not case-insensitively: exactly. And text keys arrive
dirty far more often than anybody expects, so it is worth two minutes looking first.

Every string method from the first half of this course - `.strip()`, `.lower()`, `.replace()`,
`.startswith()` - is available on a whole column at once through the **`.str` accessor**. It is
vectorized, exactly like the arithmetic on columns we did last time: no loop, one expression.

In [ ]:
print(info["name"].str.lower().head(3))
print()
print("names with whitespace to strip:", (info["name"] != info["name"].str.strip()).sum())

**Four of the 295 names have leading or trailing whitespace**, exactly as the World Bank publishes
them. `"Sub-Saharan Africa "` and `"Sub-Saharan Africa"` look identical on screen and are two
different strings.

`code` does not have the problem. Both facts matter, and we use both below.

### Merging

`.merge()` is called on one table and given the other, plus `on=` naming the shared column.

In [ ]:
panel = co2.merge(info[["code", "region", "income_level"]], on="code", how="left")

print("before:", co2.shape)
print("after: ", panel.shape)
panel[["country", "year", "co2_total", "region", "income_level"]].head(3)

Same number of rows, two extra columns. `how="left"` says *keep every row of the left-hand table,
whether or not it found a match*, and it is the right choice here: we are decorating the emissions
data, not filtering it.

The four kinds of join differ only in which unmatched rows survive:

| `how=` | Keeps |
|---|---|
| `"inner"` | only rows that matched on **both** sides - the default, and it deletes silently |
| `"left"` | every row of the left table; unmatched right-hand columns become `NaN` |
| `"right"` | every row of the right table |
| `"outer"` | everything from both sides |

Prefer `"left"` when one table is your data and the other is a lookup. Prefer `"inner"` only when you
genuinely want the intersection - and even then, count what it cost you.

### A merge can make your table bigger

If a key appears **more than once** in the right-hand table, every match produces a row.

In [ ]:
countries = pd.DataFrame({
    "code": ["NOR", "SWE"],
    "name": ["Norway", "Sweden"],
})

visits = pd.DataFrame({
    "code": ["NOR", "NOR", "NOR", "SWE"],
    "year": [2021, 2022, 2023, 2023],
})

countries.merge(visits, on="code", how="left")

Two rows in, four rows out, from a **left** join. Nothing is wrong: Norway genuinely has three
matching rows. But if you expected one row per country and got this, every average you compute
afterwards is now weighted by how many times each country happened to appear.

**A merge can lose rows and a merge can gain rows, and neither one raises anything.** So you check.

### Validating a merge

Three checks, in increasing order of how much work they save you.

**1. Count the rows.** The cheapest thing in pandas, and the one that catches most of it. Below we
deliberately merge on the dirty key to see what it costs - and because the key is called `country` in
one table and `name` in the other, we name each side with `left_on` and `right_on`.

In [ ]:
merged_on_name = co2.merge(
    info[["name", "region"]],
    left_on="country",
    right_on="name",
    how="inner",
)

print("rows before:", len(co2))
print("rows after: ", len(merged_on_name))
print("lost:       ", len(co2) - len(merged_on_name))

**48 rows gone**, deleted by the inner join without comment. 48 out of 5 904 is under one percent -
far too small to notice in a `.head()`, more than enough to be wrong about.

**2. `indicator=True` says which side each row came from.** Counting tells you *that* something went
missing; this tells you *what*. Done as a **left** join, nothing is deleted and a column called
`_merge` records what happened to each row.

In [ ]:
checked = co2.merge(
    info[["name", "region"]],
    left_on="country",
    right_on="name",
    how="left",
    indicator=True,
)

checked["_merge"].value_counts()

5 856 matched, 48 did not. **This is the diagnostic pattern worth remembering: when a merge surprises
you, redo it as a left join with `indicator=True` and look at the rows that failed.**

In [ ]:
unmatched = co2[~co2["country"].isin(info["name"])]["country"].unique()

print(list(unmatched))

Two entity names, 24 years each. And they are plainly present in the other file - you can see
`Sub-Saharan Africa` in it with your own eyes. Print the lookup file's version with `repr`, which
shows a string the way Python would write it, quotes and all.

In [ ]:
for name in info[info["name"].str.strip().isin(unmatched)]["name"]:
    print(repr(name), " length:", len(name), " stripped:", len(name.strip()))

There it is: a single trailing space, invisible in every display pandas ever produced, and it cost 48
rows. `.str.strip()` on both sides would fix it; `code` avoids it entirely. On a file with no code
column, stripping both keys before merging is the standard defense.

**3. `validate=` refuses to run a merge you did not mean.** The first two checks happen *after* the
damage. This one happens before: you state what shape the merge is supposed to be, and pandas raises
if it is not.

- `"one_to_one"` - the key is unique in both tables
- `"many_to_one"` - repeated on the left, unique on the right; the usual shape for a lookup table
- `"one_to_many"` - the other way round

Ours is many-to-one: 24 rows per country on the left, one row per country on the right.

In [ ]:
panel = co2.merge(
    info[["code", "region", "income_level"]],
    on="code",
    how="left",
    validate="many_to_one",
)

print("validated, rows:", len(panel))

In [ ]:
# MergeError
co2.merge(info[["code", "region"]], on="code", how="left", validate="one_to_one")

`Merge keys are not unique in left dataset` - because of course they are not, there are 24 rows per
country. That is the error doing its job: it caught a false belief about the data before any number
came out of it.

> 💡 **Tip:** Put `validate=` on every merge you write. It costs one argument and turns a
whole class of silent wrongness into an exception, which is the best trade available in pandas.

### One last silent one, and it belongs to `groupby`

Take the merge that failed, where 48 rows have no region, and total emissions by region for 2023.

In [ ]:
year_2023 = checked[checked["year"] == 2023]

by_region = year_2023.groupby("region")["co2_total"].sum()

print("sum of the grouped result:", by_region.sum().round(1))
print("sum of the column:        ", year_2023["co2_total"].sum().round(1))

**2 619 million tonnes have vanished between one line and the next.**

`groupby` silently drops every row whose grouping key is missing. There is a defensible reason - there
is no group called "missing" to put them in - but the effect is that a broken merge and a `groupby`
combine into a total that is quietly too small, with nothing anywhere saying so.

> ⚠️ **Warning:** After grouping, check that the parts add up to the whole. If a key can be missing,
`groupby(..., dropna=False)` keeps those rows in a group of their own rather than deleting them.

### The payoff

We now have everything needed to fix section 1. Drop the aggregates and ask the question again.

In [ ]:
countries_only = panel[panel["region"] != "Aggregates"]

print("entities:", panel["code"].nunique(), "->", countries_only["code"].nunique(), "countries")
print("rows:    ", len(panel), "->", len(countries_only))

In [ ]:
total_2023 = countries_only[countries_only["year"] == 2023]["co2_total"].sum()
world_2023 = panel[(panel["country"] == "World") & (panel["year"] == 2023)]["co2_total"].iloc[0]

print(f"sum over 203 countries: {total_2023:10,.0f}")
print(f"the World Bank's World: {world_2023:10,.0f}")

**37 483 against 39 113** - about four percent short, instead of out by a factor of eight.

The gap that remains is explicable rather than mysterious: 43 entities were aggregates, 14 more
countries have no emissions figure at all and were dropped when the file was loaded, and the World
Bank's `World` row covers territories our 203 do not. A number that is close for reasons you can name
is a much better place to be than one that matches exactly by luck.

And now the two-key `groupby` promised in section 1.

In [ ]:
countries_only.groupby(["region", "year"])["co2_total"].sum().head(3)

Two labels down the left-hand side instead of one. That is a **MultiIndex** - the same idea as before,
with the group being a *pair* rather than a single value. It is worth knowing it exists; it is usually
not worth wrestling with, and `reset_index()` turns it into two ordinary columns, which is what most
code wants.

In [ ]:
regional = countries_only.groupby(["region", "year"])["co2_total"].sum().reset_index()

regional[regional["year"] == 2023].round(0)

<div class="alert alert-info" style="background-color:#d9edf7; border-left:6px solid #31708f; border-radius:4px; padding:12px 16px; color:#31708f;">
<h3 style="margin-top:0; color:#31708f;">Your turn</h3>

<p>Starting from:</p>

<pre style="background-color:#ffffff; color:#31708f; padding:8px 10px; border-radius:3px;"><code>co2 = pd.read_csv("../data/co2_emissions.csv")
info = pd.read_csv("../data/country_info.csv")</code></pre>

<p>Merge the two tables on <code>code</code> with a left join and <code>validate=</code> set to the
right shape, drop the aggregates, and display the ten countries with the highest total emissions in
2023. Print the row count before and after the merge.</p>
</div>

## 7. Applying your own functions

Every summary so far has used a pandas method with a name in quotes: `"mean"`, `"sum"`, `"max"`.
Sooner or later a group needs an answer that is not on that list.

Here is one. What are a region's emissions **per person**? Not the average of its countries' figures -
that would let Luxembourg count as much as India. The region's total emissions divided by the region's
total population. Every built-in aggregation works on **one column**; this needs two at once, so none
of them can express it.

Last time we said `.apply()` on a column - one value in, one value out, once per row - is usually the
wrong shape, and that vectorized alternatives are shorter and faster. That argument stands. **This is
the other case**: `.apply()` on a *groupby* hands your function a whole sub-table.

In [ ]:
def emissions_per_person(group):
    """Total emissions of a group of rows, per head of its total population."""
    return group["co2_total"].sum() * 1_000_000 / group["population"].sum()


year_2023 = countries_only[countries_only["year"] == 2023]

comparison = pd.DataFrame({
    "per_person": year_2023.groupby("region").apply(emissions_per_person),
    "mean_of_countries": year_2023.groupby("region")["co2_pc"].mean(),
})

comparison.round(2)

Mostly close, and then one row that is not: the Middle East and North Africa comes out at **4.09 per
person against an unweighted mean of 9.21**. Both are arithmetically correct. The unweighted one is
dragged up by Qatar, Bahrain and Kuwait - small populations, enormous emissions per head - each
counting exactly as much as Pakistan's 248 million people.

Which one is right depends on the question: *"what does a typical country here look like?"* wants one,
*"how much does a typical person here emit?"* wants the other. What matters is knowing that you chose.

> 💡 **Tip:** The rule from last time is unchanged - reach for `np.where`, `np.select` or plain
column arithmetic first, because they are shorter and faster. `.apply()` earns its place when the
calculation needs a whole group at once, as here. It also has a row-wise form,
`df.apply(func, axis=1)`, which hands your function one row at a time - almost always replaceable by
arithmetic on whole columns, and worth recognizing rather than reaching for.

## And a look at what it bought us

We have not covered plotting, and we are not going to here. But one line is worth it, because it shows
what all this work was for: a table nobody could total correctly at the start of the session, grouped
into seven regions and followed across 24 years.

In [ ]:
east_asia = regional[regional["region"] == "East Asia & Pacific"]

east_asia.plot(x="year", y="co2_total", title="East Asia & Pacific: total CO₂, Mt")

Worth being clear about how much of this session that one line depends on. The figure is only correct
because the aggregates are gone, and they are only gone because the merge used a key that matched, and
we only know the key matched because we counted the rows. Every silent failure in this notebook would
have produced a chart that looked exactly as convincing as this one.

We come back to how to make figures worth showing somebody.

## Additional resources

- [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html) - the
  official account, including the full list of aggregation and transformation methods
- [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html) -
  every way of putting two tables together, with diagrams
- [Time series and date functionality](https://pandas.pydata.org/docs/user_guide/timeseries.html) -
  long, but the tables of `.dt` properties and of `resample` frequency codes are worth bookmarking
- [strftime.org](https://strftime.org/) - a one-page reference for every date format code
- [SKL401](https://isabelhovdahl.github.io/skl401/) - lesson 3.7 covers grouping and aggregation, 3.8
  joining, and 3.9 dates

**Next week:** turning these tables into figures - what makes a chart readable, and how to write one
function that draws the same plot for any country you hand it.